# Image classification – training

```
data/
├── training/   one folder per class (folder name = class name), e.g. pikachu/, bulbasaur/ …
├── testing/    the same class folders; optional – if missing or empty, 20% of training/ is used for testing
└── predict/    images for 2_prediction.ipynb
```

See README.md for tips on preparing images.

In [ ]:
import random
import time
from collections import Counter
from pathlib import Path

import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

TRAIN_DIR = Path("data/training")
TEST_DIR = Path("data/testing")
MODEL_PATH = "model.pt"
PRETRAINED = True  # True: ResNet-18 pretrained on ImageNet, False: small network trained from scratch
IMAGE_SIZE = 224
EPOCHS = 15
BATCH_SIZE = 64
LEARNING_RATE = 0.0003 if PRETRAINED else 0.001
TEST_FRACTION = 0.2  # used only if TEST_DIR is missing or empty
NUM_WORKERS = 8  # on a "shared memory" error, lower this (down to 0)
SEED = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
torch.multiprocessing.set_start_method("fork", force=True)  # lets DataLoader workers use load_image from this notebook
print("Device:", torch.cuda.get_device_name() if device == "cuda" else "CPU")

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)  # ImageNet colour statistics


def load_image(path):
    image = Image.open(path).convert("RGBA")
    background = Image.new("RGBA", image.size, "white")  # transparent areas become white
    return Image.alpha_composite(background, image).convert("RGB")


train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
test_tf = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


train_set = datasets.ImageFolder(TRAIN_DIR, train_tf, loader=load_image)
classes = train_set.classes

if TEST_DIR.is_dir() and any(TEST_DIR.iterdir()):
    test_set = datasets.ImageFolder(TEST_DIR, test_tf, loader=load_image)
    if test_set.classes != classes:
        raise ValueError(f"{TEST_DIR} must contain the same class folders as {TRAIN_DIR}: {classes}")
    train_targets, test_targets = train_set.targets, test_set.targets
else:
    test_idx = set()
    for c in range(len(classes)):  # the same fraction from every class
        idx = [i for i, t in enumerate(train_set.targets) if t == c]
        random.Random(SEED).shuffle(idx)
        test_idx.update(idx[:round(len(idx) * TEST_FRACTION)])
    train_idx = [i for i in range(len(train_set)) if i not in test_idx]
    train_targets = [train_set.targets[i] for i in train_idx]
    test_targets = [train_set.targets[i] for i in sorted(test_idx)]
    test_set = Subset(datasets.ImageFolder(TRAIN_DIR, test_tf, loader=load_image), sorted(test_idx))
    train_set = Subset(train_set, train_idx)
    print(f"{TEST_DIR} is missing or empty – using {TEST_FRACTION:.0%} of {TRAIN_DIR} for testing.")

train_counts, test_counts = Counter(train_targets), Counter(test_targets)
for i, name in enumerate(classes):
    print(f"{name:<30} training {train_counts[i]:5d}   testing {test_counts[i]:5d}")

In [ ]:
train_loader = DataLoader(train_set, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_set, BATCH_SIZE, num_workers=NUM_WORKERS)


def block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
        nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
        nn.MaxPool2d(2),
    )


def build_model(pretrained, num_classes):
    if pretrained:
        model = models.resnet18(weights="DEFAULT")  # downloads weights learned on 1.2 million images
        model.fc = nn.Linear(model.fc.in_features, num_classes)  # new output layer for our classes
        return model
    return nn.Sequential(
        block(3, 32), block(32, 64), block(64, 128), block(128, 256),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, num_classes),
    )


model = build_model(PRETRAINED, len(classes)).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
@torch.no_grad()
def evaluate():
    model.eval()
    correct = 0
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
    return correct / len(test_set)


for epoch in range(1, EPOCHS + 1):
    start = time.time()
    model.train()
    total_loss = correct = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast(device, dtype=torch.bfloat16, enabled=device == "cuda"):  # mixed precision
            outputs = model(x)
            loss = loss_fn(outputs, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (outputs.argmax(1) == y).sum().item()
    scheduler.step()
    print(f"Epoch {epoch:2d}/{EPOCHS} | loss {total_loss / len(train_set):.3f} | "
          f"accuracy: training {correct / len(train_set):6.1%}, testing {evaluate():6.1%} | "
          f"{time.time() - start:4.1f} s")

# besides the weights, store everything 2_prediction.ipynb needs to rebuild the same model
torch.save({"model_state": model.state_dict(), "classes": classes,
            "pretrained": PRETRAINED, "image_size": IMAGE_SIZE}, MODEL_PATH)
print("Model saved to", MODEL_PATH)